### This notebook basically focus on cleaning and preparing the dataset for training.

### 1. Dataset cleaning

### 1.1 identifying null values in bookings dataset.

In [2]:
# Cell 1: Load Dataset and Basic Information
import pandas as pd
import numpy as np

# Load the bookings dataset
bookings_df = pd.read_csv('../../datasets/bookings_train.csv')

print("=== BOOKINGS DATASET OVERVIEW ===")
print(f"Dataset shape: {bookings_df.shape}")
print(f"Columns: {list(bookings_df.columns)}")
print("\nDataset Info:")
print(bookings_df.info())
print("\nFirst 5 rows:")
print(bookings_df.head())


=== BOOKINGS DATASET OVERVIEW ===
Dataset shape: (203693, 11)
Columns: ['booking_id', 'citizen_id', 'booking_date', 'appointment_date', 'appointment_time', 'check_in_time', 'check_out_time', 'task_id', 'num_documents', 'queue_number', 'satisfaction_rating']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203693 entries, 0 to 203692
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   booking_id           203693 non-null  object
 1   citizen_id           203693 non-null  int64 
 2   booking_date         203693 non-null  object
 3   appointment_date     203693 non-null  object
 4   appointment_time     203693 non-null  object
 5   check_in_time        197601 non-null  object
 6   check_out_time       197601 non-null  object
 7   task_id              203693 non-null  object
 8   num_documents        203693 non-null  int64 
 9   queue_number         203693 non-null  int64 
 10  satisfactio

In [3]:
# Cell 2: Dataset Cleanliness Check
print("=== DATASET CLEANLINESS ANALYSIS ===")

# Check for missing values
print("\n1. Missing Values:")
missing_values = bookings_df.isnull().sum()
print(missing_values)
print(f"Total missing values: {missing_values.sum()}")

# Check for duplicates
print(f"\n2. Duplicate rows: {bookings_df.duplicated().sum()}")

# Check data types (should match the guide expectations)
print("\n3. Data Types:")
print(bookings_df.dtypes)

# Check for expected columns from the guide
expected_cols = ['booking_id', 'citizen_id', 'appointment_date', 'task_id', 'check_in_time', 'check_out_time']
print(f"\n4. Expected columns present: {all(col in bookings_df.columns for col in expected_cols)}")

# Quick statistical summary for key columns
print("\n5. Key Statistics:")
print(f"   - Unique booking_ids: {bookings_df['booking_id'].nunique()}")
print(f"   - Unique citizen_ids: {bookings_df['citizen_id'].nunique()}")
print(f"   - Unique task_ids: {bookings_df['task_id'].nunique()}")
print(f"   - Date range: {bookings_df['appointment_date'].min()} to {bookings_df['appointment_date'].max()}")

print("\n✅ Dataset cleanliness check completed!")


=== DATASET CLEANLINESS ANALYSIS ===

1. Missing Values:
booking_id                0
citizen_id                0
booking_date              0
appointment_date          0
appointment_time          0
check_in_time          6092
check_out_time         6092
task_id                   0
num_documents             0
queue_number              0
satisfaction_rating       0
dtype: int64
Total missing values: 12184

2. Duplicate rows: 0

3. Data Types:
booking_id             object
citizen_id              int64
booking_date           object
appointment_date       object
appointment_time       object
check_in_time          object
check_out_time         object
task_id                object
num_documents           int64
queue_number            int64
satisfaction_rating     int64
dtype: object

4. Expected columns present: True

5. Key Statistics:
   - Unique booking_ids: 203693
   - Unique citizen_ids: 203688
   - Unique task_ids: 19
   - Date range: 2021-01-01 to 2024-12-31

✅ Dataset cleanliness che

### Remove the null rows

In [ ]:
# Cell 4: Additional Cleaning Verification
print("=== ADDITIONAL CLEANING CHECK ===")

# 1. Check for any remaining missing values
print("1. Remaining missing values:")
print(bookings_df.isnull().sum())

# 2. First, let's examine the time data format
print("\n2. Examining time data format:")
print("Sample check_in_time values:")
print(bookings_df['check_in_time'].head(10).tolist())
print("Sample check_out_time values:")
print(bookings_df['check_out_time'].head(10).tolist())

# Clean time data - extract only time part if datetime is present
def clean_time_data(time_str):
    if pd.isna(time_str):
        return time_str
    time_str = str(time_str)
    # If it contains a date, extract just the time part
    if ' ' in time_str and len(time_str) > 10:
        return time_str.split(' ')[-1]  # Get the last part (time)
    return time_str

# Apply cleaning to time columns
bookings_df['check_in_time_clean'] = bookings_df['check_in_time'].apply(clean_time_data)
bookings_df['check_out_time_clean'] = bookings_df['check_out_time'].apply(clean_time_data)

print("\nAfter cleaning - Sample time values:")
print("check_in_time_clean:", bookings_df['check_in_time_clean'].head(5).tolist())
print("check_out_time_clean:", bookings_df['check_out_time_clean'].head(5).tolist())

# Now convert to datetime safely
try:
    bookings_df['check_in_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_in_time_clean'], errors='coerce')
    bookings_df['check_out_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_out_time_clean'], errors='coerce')
    
    # Check for conversion failures
    conversion_failures = bookings_df[bookings_df['check_in_datetime'].isna() | bookings_df['check_out_datetime'].isna()]
    print(f"\n3. Datetime conversion failures: {len(conversion_failures)}")
    
    # Calculate processing minutes
    bookings_df['processing_minutes'] = (bookings_df['check_out_datetime'] - bookings_df['check_in_datetime']).dt.total_seconds() / 60
    
    # 4. Check for invalid time logic
    invalid_times = bookings_df[bookings_df['processing_minutes'] <= 0]
    print(f"\n4. Invalid time records (check_out <= check_in): {len(invalid_times)}")
    
    # 5. Check for duplicate booking_ids
    duplicate_bookings = bookings_df['booking_id'].duplicated().sum()
    print(f"\n5. Duplicate booking_ids: {duplicate_bookings}")
    
    # 6. Check for unrealistic processing times (> 8 hours)
    extreme_times = bookings_df[bookings_df['processing_minutes'] > 480]
    print(f"\n6. Extremely long processing times (>8 hours): {len(extreme_times)}")
    if len(extreme_times) > 0:
        print(f"   Max processing time: {bookings_df['processing_minutes'].max():.1f} minutes")
    
    print(f"\n=== CLEANING SUMMARY ===")
    total_issues = len(conversion_failures) + len(invalid_times) + duplicate_bookings + len(extreme_times)
    print(f"Total additional issues found: {total_issues}")
    
    if total_issues == 0:
        print("✅ No additional cleaning required!")
    else:
        print("⚠️  Additional cleaning recommended")
        
except Exception as e:
    print(f"Error in datetime conversion: {e}")
    print("Manual inspection of time data needed")


In [4]:
# Cell 3: Data Cleaning - Remove rows with missing check-in/check-out times
print("=== DATA CLEANING ===")

print(f"Original dataset size: {len(bookings_df):,} rows")

# Remove rows where check_in_time or check_out_time is missing
# These are essential for calculating processing_minutes
bookings_clean = bookings_df.dropna(subset=['check_in_time', 'check_out_time'])

print(f"After removing missing check-in/check-out: {len(bookings_clean):,} rows")
print(f"Rows removed: {len(bookings_df) - len(bookings_clean):,}")
print(f"Data retention: {(len(bookings_clean) / len(bookings_df)) * 100:.1f}%")

# Verify no missing values in critical columns
print(f"\nMissing values after cleaning:")
print(bookings_clean[['check_in_time', 'check_out_time']].isnull().sum())

# Update our working dataset
bookings_df = bookings_clean.copy()
print(f"\n✅ Clean dataset ready: {len(bookings_df):,} rows with complete check-in/check-out data")


=== DATA CLEANING ===
Original dataset size: 203,693 rows
After removing missing check-in/check-out: 197,601 rows
Rows removed: 6,092
Data retention: 97.0%

Missing values after cleaning:
check_in_time     0
check_out_time    0
dtype: int64

✅ Clean dataset ready: 197,601 rows with complete check-in/check-out data


In [7]:


# Cell 4: Additional Cleaning Verification
print("=== ADDITIONAL CLEANING CHECK ===")

# 1. Check for any remaining missing values
print("1. Remaining missing values:")
print(bookings_df.isnull().sum())

# 2. First, let's examine the time data format
print("\n2. Examining time data format:")
print("Sample check_in_time values:")
print(bookings_df['check_in_time'].head(10).tolist())
print("Sample check_out_time values:")
print(bookings_df['check_out_time'].head(10).tolist())

# Clean time data - extract only time part if datetime is present
def clean_time_data(time_str):
    if pd.isna(time_str):
        return time_str
    time_str = str(time_str)
    # If it contains a date, extract just the time part
    if ' ' in time_str and len(time_str) > 10:
        return time_str.split(' ')[-1]  # Get the last part (time)
    return time_str

# Apply cleaning to time columns
bookings_df['check_in_time_clean'] = bookings_df['check_in_time'].apply(clean_time_data)
bookings_df['check_out_time_clean'] = bookings_df['check_out_time'].apply(clean_time_data)

print("\nAfter cleaning - Sample time values:")
print("check_in_time_clean:", bookings_df['check_in_time_clean'].head(5).tolist())
print("check_out_time_clean:", bookings_df['check_out_time_clean'].head(5).tolist())

# Now convert to datetime safely
try:
    bookings_df['check_in_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_in_time_clean'], errors='coerce')
    bookings_df['check_out_datetime'] = pd.to_datetime(bookings_df['appointment_date'] + ' ' + bookings_df['check_out_time_clean'], errors='coerce')
    
    # Check for conversion failures
    conversion_failures = bookings_df[bookings_df['check_in_datetime'].isna() | bookings_df['check_out_datetime'].isna()]
    print(f"\n3. Datetime conversion failures: {len(conversion_failures)}")
    
    # Calculate processing minutes
    bookings_df['processing_minutes'] = (bookings_df['check_out_datetime'] - bookings_df['check_in_datetime']).dt.total_seconds() / 60
    
    # 4. Check for invalid time logic
    invalid_times = bookings_df[bookings_df['processing_minutes'] <= 0]
    print(f"\n4. Invalid time records (check_out <= check_in): {len(invalid_times)}")
    
    # 5. Check for duplicate booking_ids
    duplicate_bookings = bookings_df['booking_id'].duplicated().sum()
    print(f"\n5. Duplicate booking_ids: {duplicate_bookings}")
    
    # 6. Check for unrealistic processing times (> 8 hours)
    extreme_times = bookings_df[bookings_df['processing_minutes'] > 480]
    print(f"\n6. Extremely long processing times (>8 hours): {len(extreme_times)}")
    if len(extreme_times) > 0:
        print(f"   Max processing time: {bookings_df['processing_minutes'].max():.1f} minutes")
    
    print(f"\n=== CLEANING SUMMARY ===")
    total_issues = len(conversion_failures) + len(invalid_times) + duplicate_bookings + len(extreme_times)
    print(f"Total additional issues found: {total_issues}")
    
    if total_issues == 0:
        print("✅ No additional cleaning required!")
    else:
        print("⚠️  Additional cleaning recommended")
        
except Exception as e:
    print(f"Error in datetime conversion: {e}")
    print("Manual inspection of time data needed")

=== ADDITIONAL CLEANING CHECK ===
1. Remaining missing values:
booking_id             0
citizen_id             0
booking_date           0
appointment_date       0
appointment_time       0
check_in_time          0
check_out_time         0
task_id                0
num_documents          0
queue_number           0
satisfaction_rating    0
dtype: int64

2. Examining time data format:
Sample check_in_time values:
['2021-01-01 09:11:00', '2021-01-01 09:24:00', '2021-01-01 09:29:00', '2021-01-01 10:07:00', '2021-01-01 10:26:00', '2021-01-01 10:25:00', '2021-01-01 10:29:00', '2021-01-01 10:35:00', '2021-01-01 10:34:00', '2021-01-01 13:54:00']
Sample check_out_time values:
['2021-01-01 09:48:15.166353269', '2021-01-01 10:24:12.189261137', '2021-01-01 10:26:48.802260864', '2021-01-01 11:00:13.485642822', '2021-01-01 11:54:53.260180213', '2021-01-01 12:06:10.680457034', '2021-01-01 11:45:37.549181971', '2021-01-01 11:33:33.010326653', '2021-01-01 11:34:08.857731647', '2021-01-01 14:41:52.68746630